# Task 0: Thiết lập Môi trường và Tải Dữ liệu

In [3]:
import tarfile
import os

tar_path = '../../data/lab6/hwu.tar.gz'
extract_path = '../../data/lab6/'

if os.path.exists(tar_path):
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_path)
        print(f"Extracted {tar_path} to {extract_path}")
        print("Extracted files:", tar.getnames())
else:
    print(f"File not found: {tar_path}")

print("\nContents of data/lab6/:")
for item in os.listdir('../../data/lab6'):
    print(f"  {item}")

Extracted ../../data/lab6/hwu.tar.gz to ../../data/lab6/
Extracted files: ['hwu', 'hwu/categories.json', 'hwu/train_5.csv', 'hwu/train_10.csv', 'hwu/val.csv', 'hwu/test.csv', 'hwu/train.csv']

Contents of data/lab6/:
  hwu
  hwu.tar.gz
  nlu.csv


In [4]:
import os

expected_files = ['train.csv', 'val.csv', 'test.csv']
missing_files = []

for file in expected_files:
    file_path = f'../../data/lab6/hwu/{file}'
    if os.path.exists(file_path):
        print(f"✓ Found: {file}")
    else:
        print(f"✗ Missing: {file}")
        missing_files.append(file)

if missing_files:
    print(f"\nMissing files: {missing_files}")
    print("Make sure the extraction completed successfully in the previous cell.")
else:
    print("\n✓ All expected files found! Ready to load data.")

✓ Found: train.csv
✓ Found: val.csv
✓ Found: test.csv

✓ All expected files found! Ready to load data.


In [5]:
import pandas as pd
extract_path = '../../data/lab6/hwu/'
# Dữ liệu có thể được phân tách bằng tab và không có header
df_train = pd.read_csv(f'{extract_path}train.csv', sep=',', header=None, names=['text', 'intent'])
df_val = pd.read_csv(f'{extract_path}val.csv', sep=',', header=None, names=['text', 'intent'])
df_test = pd.read_csv(f'{extract_path}test.csv', sep=',', header=None, names=['text', 'intent'])
print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8955, 2)
Validation shape: (1077, 2)
Test shape: (1077, 2)


,text,intent
0,text,category
1,what alarms do i have set right now,alarm_query
2,checkout today alarm of meeting,alarm_query
3,report alarm settings,alarm_query
4,see see for me the alarms that you have set to...,alarm_query


In [6]:
# Preprocessing intent -> numeric
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(df_train['intent'])

df_train['intent_encoded'] = encoder.transform(df_train['intent'])
df_val['intent_encoded'] = encoder.transform(df_val['intent'])
df_test['intent_encoded'] = encoder.transform(df_test['intent'])

for index, class_label in enumerate(encoder.classes_):
    print(f"{index}: '{class_label}'")
df_train.head()

0: 'alarm_query'
1: 'alarm_remove'
2: 'alarm_set'
3: 'audio_volume_down'
4: 'audio_volume_mute'
5: 'audio_volume_up'
6: 'calendar_query'
7: 'calendar_remove'
8: 'calendar_set'
9: 'category'
10: 'cooking_recipe'
11: 'datetime_convert'
12: 'datetime_query'
13: 'email_addcontact'
14: 'email_query'
15: 'email_querycontact'
16: 'email_sendemail'
17: 'general_affirm'
18: 'general_commandstop'
19: 'general_confirm'
20: 'general_dontcare'
21: 'general_explain'
22: 'general_joke'
23: 'general_negate'
24: 'general_praise'
25: 'general_quirky'
26: 'general_repeat'
27: 'iot_cleaning'
28: 'iot_coffee'
29: 'iot_hue_lightchange'
30: 'iot_hue_lightdim'
31: 'iot_hue_lightoff'
32: 'iot_hue_lighton'
33: 'iot_hue_lightup'
34: 'iot_wemo_off'
35: 'iot_wemo_on'
36: 'lists_createoradd'
37: 'lists_query'
38: 'lists_remove'
39: 'music_likeness'
40: 'music_query'
41: 'music_settings'
42: 'news_query'
43: 'play_audiobook'
44: 'play_game'
45: 'play_music'
46: 'play_podcasts'
47: 'play_radio'
48: 'qa_currency'
49: 

,text,intent,intent_encoded
0,text,category,9
1,what alarms do i have set right now,alarm_query,0
2,checkout today alarm of meeting,alarm_query,0
3,report alarm settings,alarm_query,0
4,see see for me the alarms that you have set to...,alarm_query,0


In [7]:
print("\nNumber of unique categories:", df_train['intent_encoded'].nunique())
print("\nCategory distribution:")
print(df_train['intent_encoded'].value_counts().head(10))


Number of unique categories: 65

Category distribution:
intent_encoded
2     159
56    159
49    159
25    158
42    158
29    158
20    158
6     158
0     158
18    158
Name: count, dtype: int64


# Task 1: TF-IDF + Logistic Regression

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
# 1. Tạo một pipeline với TfidfVectorizer và LogisticRegression
tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)
# 2. Train model
tfidf_lr_pipeline.fit(df_train['text'], df_train['intent_encoded'])
# 3. Đánh giá model
val_preds = tfidf_lr_pipeline.predict(df_val['text'])
print(classification_report(df_val['intent_encoded'], val_preds))


              precision    recall  f1-score   support

           0       0.78      0.95      0.86        19
           1       1.00      0.55      0.71        11
           2       0.76      0.84      0.80        19
           3       0.62      0.62      0.62         8
           4       1.00      0.53      0.70        15
           5       0.79      0.85      0.81        13
           6       0.75      0.47      0.58        19
           7       0.82      0.95      0.88        19
           8       0.88      0.79      0.83        19
           9       0.00      0.00      0.00         1
          10       0.94      0.89      0.92        19
          11       0.56      0.62      0.59         8
          12       0.82      0.74      0.78        19
          13       1.00      0.88      0.93         8
          14       0.94      0.89      0.92        19
          15       0.94      0.84      0.89        19
          16       0.86      0.95      0.90        19
          17       1.00    

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_clas

In [43]:
# In ra "kiến trúc" của pipeline
print(tfidf_lr_pipeline)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(max_features=5000)),
                ('logisticregression', LogisticRegression(max_iter=1000))])


# Task 2: Pipeline Word2Vec (Trung bình) + Dense Layer

In [10]:
%load_ext tensorboard

In [11]:
from tensorflow.keras.callbacks import TensorBoard
import datetime

log_dir = "../../results/lab6/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# 2. Tạo callback
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [12]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

In [13]:
# 1. Huấn luyện mô hình Word2Vec trên dữ liệu text của bạn
train_sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(train_sentences, vector_size=100, window=5, min_count=1, workers=4)

In [14]:
train_sentences[:5]

[['text'],
 ['what', 'alarms', 'do', 'i', 'have', 'set', 'right', 'now'],
 ['checkout', 'today', 'alarm', 'of', 'meeting'],
 ['report', 'alarm', 'settings'],
 ['see',
  'see',
  'for',
  'me',
  'the',
  'alarms',
  'that',
  'you',
  'have',
  'set',
  'tomorrow',
  'morning']]

In [15]:
# 2. Viết hàm để chuyển mỗi câu thành vector trung bình
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))
def sentence_to_avg_vector(text, model):
    vector_size = model.vector_size
    words = text.split()
    word_vectors = []
    for word in words:
        word_lower = word.lower()
        if word_lower in STOP_WORDS:
            continue
        if word_lower in model.wv:  # model.wv chứa tất cả các vector từ
            word_vectors.append(model.wv[word_lower])
    if not word_vectors:
        return np.zeros(vector_size, dtype=np.float32)
    avg_vector = np.mean(word_vectors, axis=0)
    return avg_vector

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
# 3. Chuyển đổi tất cả các câu trong tập train, val, test thành vectors
X_train = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])

y_train = df_train['intent_encoded'].values
y_val = df_val['intent_encoded'].values
y_test = df_test['intent_encoded'].values

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (8955, 100)
X_val shape: (1077, 100)
X_test shape: (1077, 100)
y_train shape: (8955,)


In [17]:
# 4. Xây dựng mô hình Sequential của Keras (3 hidden layers)
num_classes = len(encoder.classes_)
model = Sequential([
    Dense(256, activation='relu', input_shape=(w2v_model.vector_size,)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu', kernel_regularizer='l2'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),

    Dense(num_classes, activation='softmax')
])

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
from tensorflow.keras.optimizers import Adam

# 5.1. Compile
model.compile(loss='sparse_categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        25,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 65)             │         4,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 73,025 (285.25 KB)

 Trainable params: 72,129 (281.75 KB)

 Non-trainable params: 896 (3.50 KB)

In [19]:
# 5.2. Huấn luyện + đánh giá
from tensorflow.keras.callbacks import EarlyStopping
history = model.fit(X_train, y_train,
                    epochs=50, # Thử 50 epochs
                    batch_size=256,
                    validation_data=(X_val, y_val),
                    callbacks = [EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True), tensorboard_callback],
                    verbose=1)

print("\n--- Đánh giá trên tập Test ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.0614 - loss: 5.5381 - val_accuracy: 0.0334 - val_loss: 5.2102
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1280 - loss: 4.4687 - val_accuracy: 0.0334 - val_loss: 4.8328
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2102 - loss: 3.7821 - val_accuracy: 0.0362 - val_loss: 4.6224
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2891 - loss: 3.2571 - val_accuracy: 0.0176 - val_loss: 4.5037
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3367 - loss: 2.9129 - val_accuracy: 0.0223 - val_loss: 4.4414
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3851 - loss: 2.6241 - val_accuracy: 0.0316 - val_loss: 4.3811
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4213 - loss: 2.4350 - val_accuracy: 0.0316 - val_loss: 4.2941
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4293 - loss: 2.3047 - val_accuracy: 0.0316 - v

In [20]:
print("\n--- Đánh giá chi tiết trên tập Test ---")

y_pred_probabilities = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probabilities, axis=1)
y_true_classes = y_test 
target_names = encoder.classes_

print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))


--- Đánh giá chi tiết trên tập Test ---
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
                          precision    recall  f1-score   support

             alarm_query       1.00      0.58      0.73        19
            alarm_remove       0.88      0.64      0.74        11
               alarm_set       0.52      0.63      0.57        19
       audio_volume_down       0.00      0.00      0.00         8
       audio_volume_mute       0.50      0.53      0.52        15
         audio_volume_up       0.50      0.62      0.55        13
          calendar_query       0.09      0.05      0.07        19
         calendar_remove       0.77      0.89      0.83        19
            calendar_set       0.25      0.53      0.34        19
                category       0.00      0.00      0.00         1
          cooking_recipe       0.39      0.47      0.43        19
        datetime_convert       0.71      0.62      0.67         8
          datetime_query       0.46      0.68      0.55      

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_clas

# Task 3: Embedding Pre-trained + LSTM

In [21]:
! pip install tensorflow.keras

In [22]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional

In [23]:
# 1. Tiền xử lý cho mô hình chuỗi
## 1.1 Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
vocab_size = 5000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])
train_sequences = tokenizer.texts_to_sequences(df_train['text'])
val_sequences = tokenizer.texts_to_sequences(df_val['text'])
test_sequences = tokenizer.texts_to_sequences(df_test['text'])

In [24]:
print(train_sequences[:5])

[[920], [9, 99, 24, 5, 26, 35, 92, 62], [809, 39, 36, 15, 113], [606, 36, 532], [265, 265, 12, 4, 2, 99, 27, 8, 26, 35, 89, 138]]


In [25]:
max = np.max([len(seq) for seq in train_sequences])
print(max)

25


In [26]:
# 1.2. Padding: Đảm bảo các chuỗi có cùng độ dài
max_len = 30
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post')
# ... (Tương tự cho val và test)
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_sequences, maxlen=max_len, padding='post')

In [27]:
# 2. Tạo ma trận trọng số cho Embedding Layer từ Word2Vec
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print(embedding_matrix.shape)
print(max_len)
print(vocab_size)

(4265, 100)
30
4265


In [28]:
# 3. Xây dựng mô hình Sequential với LSTM
max_len = 30
vocab_size = len(tokenizer.word_index) + 1
lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False  # Đóng băng embedding layer
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0),
    Dense(num_classes, activation='softmax')
])
lstm_model_pretrained.build(input_shape=(None, max_len))
lstm_model_pretrained.compile(loss='sparse_categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])
print(lstm_model_pretrained.summary())

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 100)        │       426,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 552,133 (2.11 MB)

 Trainable params: 125,633 (490.75 KB)

 Non-trainable params: 426,500 (1.63 MB)

None


In [29]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_model_pretrained.fit(X_train_pad, y_train,
                    epochs=50, 
                    batch_size=256,
                    validation_data=(X_val_pad, y_val),
                    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True), tensorboard_callback]
                   )

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.0191 - loss: 4.1432 - val_accuracy: 0.0334 - val_loss: 4.0868
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.0350 - loss: 4.0661 - val_accuracy: 0.0381 - val_loss: 4.0045
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.0366 - loss: 4.0025 - val_accuracy: 0.0474 - val_loss: 3.9036
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.0515 - loss: 3.8944 - val_accuracy: 0.0566 - val_loss: 3.8216
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.0521 - loss: 3.8343 - val_accuracy: 0.0678 - val_loss: 3.7507
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.0577 - loss: 3.7929 - val_accuracy: 0.0650 - val_loss: 3.7337
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.0672 - loss: 3.7203 - val_accuracy: 0.0687 - val_loss: 3.6122
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.0784 - loss: 3.6327 - val_accuracy: 0.1049 - v

In [40]:
# Test trên tập test
loss, accuracy = lstm_model_pretrained.evaluate(X_test_pad, y_test, verbose=1)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

y_pred_probabilities = lstm_model_pretrained.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probabilities, axis=1)

target_names = encoder.classes_
print(classification_report(y_test, y_pred_classes, 
                            target_names=target_names, 
                            zero_division=0))

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2730 - loss: 2.6828
Test Loss: 2.6828
Test Accuracy: 0.2730
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
                          precision    recall  f1-score   support

             alarm_query       0.63      0.63      0.63        19
            alarm_remove       0.73      0.73      0.73        11
               alarm_set       0.89      0.84      0.86        19
       audio_volume_down       0.50      0.12      0.20         8
       audio_volume_mute       0.18      0.13      0.15        15
         audio_volume_up       0.00      0.00      0.00        13
          calendar_query       0.00      0.00      0.00        19
         calendar_remove       0.35      0.42      0.38        19
            calendar_set       0.17      0.26      0.20        19
                category       0.00      0.00      0.00         1
          cooking_recipe       0.14      0.11      0.12        19
        datetime_convert       0.50      0.38      0.4

# Task 4: Embedding học từ đầu + LSTM

In [35]:
# 1. Xây dựng mô hình
lstm_model_scratch = Sequential([
    Embedding(vocab_size, 200, input_length=max_len),
    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

In [36]:
# 2. Compile, huấn luyện (sử dụng EarlyStopping) và đánh giá
lstm_model_scratch.compile(loss='sparse_categorical_crossentropy',
                optimizer=Adam(learning_rate=0.001),
                metrics=['accuracy'])

lstm_model_scratch.build(input_shape=(None, max_len))
lstm_model_scratch.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 30, 200)        │       853,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       168,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,030,345 (3.93 MB)

 Trainable params: 1,030,089 (3.93 MB)

 Non-trainable params: 256 (1.00 KB)

In [37]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)
callbacks_list = [early_stopping, tensorboard_callback]

In [38]:
history = lstm_model_scratch.fit(
    X_train_pad, y_train,
    epochs=50,            
    batch_size=256,
    validation_data=(X_val_pad, y_val),
    callbacks=callbacks_list,  
    verbose=1
    )

Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 0.0150 - loss: 4.2510 - val_accuracy: 0.0176 - val_loss: 4.1664
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0237 - loss: 4.1615 - val_accuracy: 0.0362 - val_loss: 4.1558
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0710 - loss: 3.7621 - val_accuracy: 0.2396 - val_loss: 4.0115
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.1648 - loss: 3.1493 - val_accuracy: 0.4002 - val_loss: 3.6735
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.2288 - loss: 2.7267 - val_accuracy: 0.5348 - val_loss: 2.9944
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.3040 - loss: 2.3673 - val_accuracy: 0.5812 - val_loss: 2.4696
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.3592 - loss: 2.0816 - val_accuracy: 0.6230 - val_loss: 1.9678
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.4409 - loss: 1.7957 - val_accuracy: 0.6806 - v

In [41]:
loss, accuracy = lstm_model_scratch.evaluate(X_test_pad, y_test, verbose=0)

print(f"Test Loss    : {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({(accuracy*100):.2f}%)")

y_pred_probabilities = lstm_model_scratch.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probabilities, axis=1)

target_names = encoder.classes_
print(classification_report(y_test, y_pred_classes, 
                            target_names=target_names, 
                            zero_division=0))

Test Loss    : 0.7899
Test Accuracy: 0.8078 (80.78%)
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
                          precision    recall  f1-score   support

             alarm_query       0.85      0.89      0.87        19
            alarm_remove       1.00      0.91      0.95        11
               alarm_set       0.82      0.95      0.88        19
       audio_volume_down       0.75      0.75      0.75         8
       audio_volume_mute       1.00      0.80      0.89        15
         audio_volume_up       0.81      1.00      0.90        13
          calendar_query       0.35      0.42      0.38        19
         calendar_remove       0.76      1.00      0.86        19
            calendar_set       0.76      0.68      0.72        19
                category       0.00      0.00      0.00         1
          cooking_recipe       0.75      0.63      0.69        19
        datetime_convert       0.70      0.88      0.78         8
          datetime_query       0.80      0.84    

# Task 5: Đánh giá, So sánh và Phân tích

In [45]:
tensorboard --logdir log_dir

Reusing TensorBoard on port 6006 (pid 29328), started 9:30:15 ago. (Use '!kill 29328' to kill it.)

In [49]:
# Tổng hợp kết quả các model bao gồm loss
from sklearn.metrics import accuracy_score, classification_report, log_loss
import numpy as np

print("=== TỔNG HỢP KẾT QUẢ CÁC MODEL (bao gồm Loss) ===\n")

# 1. TF-IDF + LogReg (từ validation set)
print("1. TF-IDF + Logistic Regression:")
val_preds_tfidf = tfidf_lr_pipeline.predict(df_val['text'])
val_proba_tfidf = tfidf_lr_pipeline.predict_proba(df_val['text'])
val_accuracy_tfidf = accuracy_score(df_val['intent_encoded'], val_preds_tfidf)
val_loss_tfidf = log_loss(df_val['intent_encoded'], val_proba_tfidf)
report_tfidf = classification_report(df_val['intent_encoded'], val_preds_tfidf, output_dict=True)
print(f"   Validation Accuracy: {val_accuracy_tfidf:.4f}")
print(f"   Validation Loss: {val_loss_tfidf:.4f}")
print(f"   Macro F1-score: {report_tfidf['macro avg']['f1-score']:.4f}")
print(f"   Weighted F1-score: {report_tfidf['weighted avg']['f1-score']:.4f}")

# 2. Word2Vec + Dense NN (đã có kết quả từ cell trước)
print("\n2. Word2Vec + Dense NN:")
print(f"   Test Accuracy: 0.5600")  # từ output trước
print(f"   Test Loss: N/A")  # không có loss được lưu
print(f"   Macro F1-score: 0.5100") # từ output trước
print(f"   Weighted F1-score: 0.5400") # từ output trước

# 3. Pre-trained Word2Vec + LSTM
print("\n3. Pre-trained Word2Vec + LSTM:")
test_loss_pretrained, test_acc_pretrained = lstm_model_pretrained.evaluate(X_test_pad, y_test, verbose=0)
print(f"   Test Accuracy: {test_acc_pretrained:.4f}")
print(f"   Test Loss: {test_loss_pretrained:.4f}")
print(f"   Macro F1-score: 0.2300") # từ output trước
print(f"   Weighted F1-score: 0.2400") # từ output trước

# 4. Embedding từ đầu + LSTM
print("\n4. Embedding từ đầu + LSTM:")
test_loss_scratch, test_acc_scratch = lstm_model_scratch.evaluate(X_test_pad, y_test, verbose=0)
print(f"   Test Accuracy: {test_acc_scratch:.4f}")
print(f"   Test Loss: {test_loss_scratch:.4f}")
print(f"   Macro F1-score: 0.7900") # từ output trước
print(f"   Weighted F1-score: 0.8100") # từ output trước

print("\n=== SUMMARY TABLE FOR REPORT ===")
print("| Model | Accuracy | Loss | Macro F1 | Weighted F1 |")
print("|-------|----------|------|----------|-------------|")
print(f"| TF-IDF+LogReg | {val_accuracy_tfidf:.4f} | {val_loss_tfidf:.4f} | {report_tfidf['macro avg']['f1-score']:.4f} | {report_tfidf['weighted avg']['f1-score']:.4f} |")
print(f"| Word2Vec+Dense | 0.5600 | N/A | 0.5100 | 0.5400 |")
print(f"| Pretrained+LSTM | {test_acc_pretrained:.4f} | {test_loss_pretrained:.4f} | 0.2300 | 0.2400 |")
print(f"| Scratch+LSTM | {test_acc_scratch:.4f} | {test_loss_scratch:.4f} | 0.7900 | 0.8100 |")

=== TỔNG HỢP KẾT QUẢ CÁC MODEL (bao gồm Loss) ===

1. TF-IDF + Logistic Regression:
   Validation Accuracy: 0.8589
   Validation Loss: 0.9800
   Macro F1-score: 0.8251
   Weighted F1-score: 0.8567

2. Word2Vec + Dense NN:
   Test Accuracy: 0.5600
   Test Loss: N/A
   Macro F1-score: 0.5100
   Weighted F1-score: 0.5400

3. Pre-trained Word2Vec + LSTM:


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_clas

   Test Accuracy: 0.2730
   Test Loss: 2.6828
   Macro F1-score: 0.2300
   Weighted F1-score: 0.2400

4. Embedding từ đầu + LSTM:
   Test Accuracy: 0.8078
   Test Loss: 0.7899
   Macro F1-score: 0.7900
   Weighted F1-score: 0.8100

=== SUMMARY TABLE FOR REPORT ===
| Model | Accuracy | Loss | Macro F1 | Weighted F1 |
|-------|----------|------|----------|-------------|
| TF-IDF+LogReg | 0.8589 | 0.9800 | 0.8251 | 0.8567 |
| Word2Vec+Dense | 0.5600 | N/A | 0.5100 | 0.5400 |
| Pretrained+LSTM | 0.2730 | 2.6828 | 0.2300 | 0.2400 |
| Scratch+LSTM | 0.8078 | 0.7899 | 0.7900 | 0.8100 |
   Test Accuracy: 0.8078
   Test Loss: 0.7899
   Macro F1-score: 0.7900
   Weighted F1-score: 0.8100

=== SUMMARY TABLE FOR REPORT ===
| Model | Accuracy | Loss | Macro F1 | Weighted F1 |
|-------|----------|------|----------|-------------|
| TF-IDF+LogReg | 0.8589 | 0.9800 | 0.8251 | 0.8567 |
| Word2Vec+Dense | 0.5600 | N/A | 0.5100 | 0.5400 |
| Pretrained+LSTM | 0.2730 | 2.6828 | 0.2300 | 0.2400 |
| Scratch+L

In [52]:
# Thêm vài câu ví dụ "khó" / thường gặp vào danh sách user_examples
user_examples = [
    ("can you remind me to not call my mom", "reminder_create"),
    ("is it going to be sunny or rainy tomorrow", "weather_query"),
    ("find a flight from new york to london but not through paris", "flight_search"),
    ("Can you please set an alarm for me tomorrow morning at 6 AM", "alarm_set"),
    ("Don't play music right now", "general_commandstop"),
    ("Weather today", "weather_query"),
]

# Hàm tiện lợi để in top-k dự đoán (label, prob)
def topk_from_probas(probas, k=3):
    idxs = np.argsort(probas)[::-1][:k]
    labels = encoder.inverse_transform(idxs)
    return [(labels[i], float(probas[idxs[i]])) for i in range(len(idxs))]

for text, true_label in user_examples:
    print("="*80)
    print(f"Input: {text!r}")
    # True label -> index (nếu có)
    try:
        true_idx = int(encoder.transform([true_label])[0])
    except Exception:
        true_idx = None
    print(f"True label: {true_label} (encoded: {true_idx})")

    # 1) TF-IDF + LogisticRegression
    tf_pred_idx = int(tfidf_lr_pipeline.predict([text])[0])
    tf_proba = tfidf_lr_pipeline.predict_proba([text])[0]
    tf_pred_label = encoder.inverse_transform([tf_pred_idx])[0]
    tf_pred_prob = float(tf_proba[tf_pred_idx])
    tf_top3 = topk_from_probas(tf_proba, k=3)
    print(f"TF-IDF+LR -> {tf_pred_label} (idx {tf_pred_idx}, prob {tf_pred_prob:.4f})", "OK" if true_idx==tf_pred_idx else "WRONG")
    print(f"   top3: {tf_top3}")

    # 2) Word2Vec (avg) + Dense NN (variable name: model)
    vec = sentence_to_avg_vector(text, w2v_model)
    w2v_proba = model.predict(np.array([vec]), verbose=0)[0]
    w2v_idx = int(np.argmax(w2v_proba))
    w2v_label = encoder.inverse_transform([w2v_idx])[0]
    w2v_prob = float(w2v_proba[w2v_idx])
    w2v_top3 = topk_from_probas(w2v_proba, k=3)
    print(f"Word2Vec+Dense -> {w2v_label} (idx {w2v_idx}, prob {w2v_prob:.4f})", "OK" if true_idx==w2v_idx else "WRONG")
    print(f"   top3: {w2v_top3}")

    # 3) Pretrained Word2Vec embedding + LSTM
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    pre_proba = lstm_model_pretrained.predict(pad, verbose=0)[0]
    pre_idx = int(np.argmax(pre_proba))
    pre_label = encoder.inverse_transform([pre_idx])[0]
    pre_prob = float(pre_proba[pre_idx])
    pre_top3 = topk_from_probas(pre_proba, k=3)
    print(f"PretrainedEmbedding+LSTM -> {pre_label} (idx {pre_idx}, prob {pre_prob:.4f})", "OK" if true_idx==pre_idx else "WRONG")
    print(f"   top3: {pre_top3}")

    # 4) Embedding (scratch) + LSTM
    pad2 = pad  # reuse pad
    scr_proba = lstm_model_scratch.predict(pad2, verbose=0)[0]
    scr_idx = int(np.argmax(scr_proba))
    scr_label = encoder.inverse_transform([scr_idx])[0]
    scr_prob = float(scr_proba[scr_idx])
    scr_top3 = topk_from_probas(scr_proba, k=3)
    print(f"ScratchEmbedding+LSTM -> {scr_label} (idx {scr_idx}, prob {scr_prob:.4f})", "OK" if true_idx==scr_idx else "WRONG")
    print(f"   top3: {scr_top3}")


Input: 'can you remind me to not call my mom'
True label: reminder_create (encoded: None)
TF-IDF+LR -> calendar_set (idx 8, prob 0.3335) WRONG
   top3: [('calendar_set', 0.333538858079335), ('email_querycontact', 0.07068064281273394), ('general_negate', 0.053637147084640194)]
Word2Vec+Dense -> calendar_set (idx 8, prob 0.3210) WRONG
   top3: [('calendar_set', 0.3210252821445465), ('email_sendemail', 0.22479769587516785), ('email_query', 0.11238753795623779)]
PretrainedEmbedding+LSTM -> social_post (idx 56, prob 0.0986) WRONG
   top3: [('social_post', 0.09857174009084702), ('takeaway_order', 0.07268866151571274), ('recommendation_locations', 0.05791664123535156)]
ScratchEmbedding+LSTM -> calendar_set (idx 8, prob 0.9835) WRONG
   top3: [('calendar_set', 0.9834993481636047), ('alarm_set', 0.005007795058190823), ('email_querycontact', 0.003914648201316595)]
Input: 'is it going to be sunny or rainy tomorrow'
True label: weather_query (encoded: 64)
TF-IDF+LR -> weather_query (idx 64, prob 0

In [53]:
print("Kiểm tra quick:")
print("num_classes variable:", num_classes)
print("encoder.classes_ length:", len(encoder.classes_))
print("Test set intent distribution:")
print(df_test['intent'].value_counts())

Kiểm tra quick:
num_classes variable: 65
encoder.classes_ length: 65
Test set intent distribution:
intent
weather_query          19
general_repeat         19
iot_hue_lightchange    19
transport_traffic      19
alarm_query            19
                       ..
email_addcontact        8
music_settings          7
iot_wemo_on             7
iot_hue_lighton         3
category                1
Name: count, Length: 65, dtype: int64
